In [ ]:
import numpy as np
import cv2
import matplotlib.pyplot as plt

file_path = "/home/hadis/custom_vector/buildParticleDot/buildVS/nov12_velocityVerlet/N_particle_PosMD.dat"
particle_type = "dot"

skip_rows = 50
frame_size = 800
particle_radius = 10
output_file = 'dot_nov12_vs_vv.mp4'

freedom_d = 2 if particle_type == "dot" else 3

positions = []
with open(file_path, 'r') as file:
    for i, line in enumerate(file):
        if i % skip_rows == 0:
            data = line.strip().split()
            data = list(map(float, data[1:]))
            positions.append(data)
positions = np.array(positions)

fourcc = cv2.VideoWriter_fourcc(*'mp4v')
out = cv2.VideoWriter(output_file, fourcc, 30.0, (frame_size, frame_size))

scale = frame_size / 20
offset = 0

def map_to_frame(x, y):
    return int(x * scale + offset), int(y * scale + offset)

for i, frame_data in enumerate(positions):
    frame = np.ones((frame_size, frame_size, 3), dtype=np.uint8) * 255

    for j in range(0, len(frame_data), freedom_d):
        x, y = frame_data[j], frame_data[j + 1]
        cx, cy = map_to_frame(x, y)
        color = (120, 120, 120)
        cv2.circle(frame, (cx, cy), particle_radius, color, -1)

        if freedom_d == 3:
            phi = frame_data[j + 2]
            line_length = 10
            end_x = int(cx + line_length * np.cos(np.radians(phi)))
            end_y = int(cy + line_length * np.sin(np.radians(phi)))
            cv2.line(frame, (cx, cy), (end_x, end_y), (0, 0, 255), 2)

    out.write(frame)

    if i % 50 == 0:
        print(f"Processing frame {i}/{len(positions)}")

out.release()
print("Video saved as", output_file)

plt.imshow(cv2.cvtColor(frame, cv2.COLOR_BGR2RGB))
plt.title("Particle Movement at Last Frame")
plt.show()
